In [1]:
#Q1: What is the OS release?

In [2]:
with open("/etc/os-release", "r") as f:
    os_release = f.read()

os_release

'PRETTY_NAME="Ubuntu 24.04.4 LTS"\nNAME="Ubuntu"\nVERSION_ID="24.04"\nVERSION="24.04.4 LTS (Noble Numbat)"\nVERSION_CODENAME=noble\nID=ubuntu\nID_LIKE=debian\nHOME_URL="https://www.ubuntu.com/"\nSUPPORT_URL="https://help.ubuntu.com/"\nBUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"\nPRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"\nUBUNTU_CODENAME=noble\nLOGO=ubuntu-logo\n'

In [3]:
#Q2: What CPUs does your VM have?

In [4]:
import subprocess

cpu_info = subprocess.check_output(["lscpu"], text=True)

cpu_info

'Architecture:                            x86_64\nCPU op-mode(s):                          32-bit, 64-bit\nAddress sizes:                           46 bits physical, 48 bits virtual\nByte Order:                              Little Endian\nCPU(s):                                  2\nOn-line CPU(s) list:                     0,1\nVendor ID:                               GenuineIntel\nModel name:                              Intel(R) Xeon(R) CPU @ 2.20GHz\nCPU family:                              6\nModel:                                   79\nThread(s) per core:                      2\nCore(s) per socket:                      1\nSocket(s):                               1\nStepping:                                0\nBogoMIPS:                                4400.32\nFlags:                                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc rep_good nopl xtopology nonstop_tsc cpuid

In [5]:
import os
import pyarrow.fs as fs

P8_BUCKET = "cs544_praneethgorrepati"

gcs = fs.GcsFileSystem()

local_file = "../wi-schools-raw.parquet"
gcs_path = f"{P8_BUCKET}/wi-schools-raw.parquet"

with open(local_file, "rb") as f, gcs.open_output_stream(gcs_path) as out:
    while True:
        chunk = f.read(1024 * 1024)
        if not chunk:
            break
        out.write(chunk)

In [6]:
#Q3: What are the paths of all the files/directories in the top-level of your bucket?

In [7]:
selector = fs.FileSelector(P8_BUCKET, recursive=False)
infos = gcs.get_file_info(selector)

q3 = [info.path for info in infos]
q3

['cs544_praneethgorrepati/wi-schools-raw.parquet']

In [8]:
#Q4: What is the modification time of wi-schools-raw.parquet, in nanoseconds?

In [9]:
file_info = gcs.get_file_info(f"{P8_BUCKET}/wi-schools-raw.parquet")
file_info
q4 = file_info.mtime_ns
q4

1777508000399000000

In [72]:
wi_counties_sqlx = """
config {
  type: "table",
  name: "wi_counties"
}

SELECT *
FROM `bigquery-public-data.geo_us_boundaries.counties`
WHERE state_fips_code = "55"
"""

In [73]:
schools_sqlx = """
config {
  type: "operations",
  name: "schools",
  hasOutput: true
}

LOAD DATA OVERWRITE `...p8.schools`
FROM FILES (
  format = 'PARQUET',
  uris = ['gs://cs544_praneethgorrepati/wi-schools-raw.parquet']
)
"""

In [74]:
wi_county_schools_sqlx = """
config {
  type: "table",
  name: "wi_county_schools"
}

SELECT
  s.*,
  ST_GEOGPOINT(s.longitude, s.latitude) AS location,
  c.county_name
FROM ${ref("schools")} s
JOIN ${ref("wi_counties")} c
ON ST_CONTAINS(c.county_geom, ST_GEOGPOINT(s.longitude, s.latitude))
"""

In [79]:
from google.cloud import dataform_v1beta1 as dataform

client = dataform.DataformClient()

PROJECT_ID = "gen-lang-client-0862809283"
REPOSITORY_ID = "cs544p8"
WORKSPACE_ID = "p8workspace"

REPOSITORY_PATH = f"projects/{PROJECT_ID}/locations/us-central1/repositories/{REPOSITORY_ID}"
WORKSPACE_PATH = f"{REPOSITORY_PATH}/workspaces/{WORKSPACE_ID}"

def upload_file(filename, content):
    request = dataform.WriteFileRequest(
        workspace=WORKSPACE_PATH,
        path=f"definitions/{filename}",
        contents=content.encode("utf-8")
    )
    client.write_file(request=request)

upload_file("wi_counties.sqlx", wi_counties_sqlx)
upload_file("schools.sqlx", schools_sqlx)
upload_file("wi_county_schools.sqlx", wi_county_schools_sqlx)

compilation = client.create_compilation_result(
    dataform.CreateCompilationResultRequest(
        parent=REPOSITORY_PATH,
        compilation_result=dataform.CompilationResult(
            workspace=WORKSPACE_PATH
        ),
    )
)

if compilation.compilation_errors:
    print(compilation.compilation_errors)
else:
    print("Compilation successful!")


Compilation successful!


In [80]:
#Q5: What are the dependencies between all the Dataform tables?

In [81]:
compilation = client.create_compilation_result(
    dataform.CreateCompilationResultRequest(
        parent=REPOSITORY_PATH,
        compilation_result=dataform.CompilationResult(
            workspace=WORKSPACE_PATH
        ),
    )
)

compilation_result_name = compilation.name
compilation_result_name

response = client.query_compilation_result_actions(
    request={"name": compilation_result_name}
)

deps = {}

for action in response.compilation_result_actions:
    name = action.target.name
    depends = [d.name for d in action.relation.dependency_targets]
    deps[name] = depends

deps

{'schools': [],
 'wi_counties': [],
 'wi_county_schools': ['schools', 'wi_counties']}

In [87]:
#Q6: How many counties are in Wisconsin?

In [100]:
from google.cloud import bigquery

bq = bigquery.Client(project=project_id)

query = """
SELECT COUNT(*) AS num_counties
FROM `gen-lang-client-0862809283.p8.wi_counties`
"""

result = bq.query(query).result()
row = list(result)[0]

int(row["num_counties"])

72

In [101]:
#Q7: How many public schools are there in Wisconsin?

In [105]:
query = """
SELECT COUNT(*) AS num_public_schools
FROM `gen-lang-client-0862809283.p8.wi_county_schools`
WHERE agency_type = "Public school"
"""

result = bq.query(query).result()
row = list(result)[0]
int(row["num_public_schools"])

2116

In [106]:
#Q8: Which counties have at least 2 cities that each contain 3 or more public high schools?

In [109]:
query = """
FROM `gen-lang-client-0862809283.p8.wi_county_schools`
|> WHERE agency_type = "Public school"
|> WHERE school_type = "High School"
|> AGGREGATE COUNT(*) AS school_count
   GROUP BY county_name, city
|> WHERE school_count >= 3
|> AGGREGATE COUNT(DISTINCT city) AS city_count
   GROUP BY county_name
|> WHERE city_count >= 2
|> SELECT county_name
"""

df = bq.query(query).to_dataframe()
df["county_name"].tolist()

['Waukesha', 'Milwaukee', 'Dane', 'Brown']

In [68]:
#Q9 How many times can you run the query from Q8 before you exhaust the BigQuery free tier (1 TiB of processed data)?

In [110]:
job = bq.query(query)
job.result()

bytes_processed = job.total_bytes_processed
bytes_processed

TI_B = 1024 ** 4  # 1 TiB in bytes

runs = TI_B 
runs

1099511627776

In [111]:
#Q10: Which MMSD elementary schools are named after people/places/etc?

In [115]:
query = """
SELECT school_name
FROM `gen-lang-client-0862809283.p8.wi_county_schools`
WHERE district = "Madison Metropolitan"
  AND agency_type = "Public school"
  AND school_type = "Elementary School"
"""

df = bq.query(query).to_dataframe()
df.head()

def clean_name(name):
    return (
        name.replace("Elementary School", "")
            .replace("Elementary", "")
            .strip()
    )

df["clean_name"] = df["school_name"].apply(clean_name)
df["clean_name"].tolist()

query = """
SELECT
  school_name,
  AI.CLASSIFY(
    REPLACE(
      REPLACE(school_name, "Elementary School", ""),
      "Elementary",
      ""
    ),
    ["Person", "Place", "Other"]
  ) AS category
FROM `gen-lang-client-0862809283.p8.wi_county_schools`
WHERE district = "Madison Metropolitan"
  AND agency_type = "Public school"
  AND school_type = "Elementary School"
"""

df = bq.query(query).to_dataframe()

result_dict = dict(zip(df["school_name"], df["category"]))
result_dict

{'Schenk Elementary': 'Person',
 'Franklin Elementary': 'Person',
 'Lapham Elementary': 'Person',
 'Randall Elementary School': 'Person',
 'Muir Elementary': 'Person',
 'Southside Elementary School': 'Place',
 'Lincoln Elementary': 'Person',
 '4K PK Off Site': 'Other',
 'Mendota Elementary': 'Place',
 'Sandburg Elementary': 'Person',
 'Stephens Elementary': 'Person',
 'Kennedy Elementary': 'Person',
 'Crestwood Elementary': 'Place',
 'Gompers Elementary': 'Person',
 'Shorewood Hills Elementary': 'Place',
 'Thoreau Elementary': 'Person',
 'Huegel Elementary': 'Person',
 'Dr Virginia Henderson Elementary School': 'Person',
 'Olson Elementary': 'Person',
 'Marquette Elementary': 'Place',
 'Lake View Elementary': 'Place',
 'Lowell Elementary': 'Place',
 'Van Hise Elementary': 'Person',
 'Midvale Elementary': 'Place',
 'Milele Chikasa Anana Elementary School': 'Person',
 'Orchard Ridge Elementary': 'Place',
 'Emerson Elementary': 'Person',
 'Leopold Elementary': 'Person',
 'Hawthorne Elemen